# Real / Fake Job Posting Prediction

**Dataset:** [Real or Fake: Fake Job Posting Prediction](https://www.kaggle.com/datasets/shivamb/real-or-fake-fake-jobposting-prediction) (Kaggle)

## The learning problem

Each row is a single online job advertisement described by a mix of free text
(`title`, `company_profile`, `description`, `requirements`, `benefits`), categorical
metadata (`employment_type`, `required_experience`, `required_education`, `industry`,
`function`, `location`) and a few binary flags (`telecommuting`, `has_company_logo`,
`has_questions`). The target `fraudulent` marks the ad as a scam (1) or a genuine
posting (0). This is **supervised binary classification on text**.

The dataset has 17,880 postings of which only ~866 (**4.8%**) are fraudulent, so it is
strongly imbalanced. A model that predicts "real" for everything would score 95%
accuracy while catching zero scams — which is why every metric below is reported for
the **fraudulent class only**, and why the grid search is scored on that class's F1.

## What this notebook does

1. Load and clean the data (drop `job_id`, fill missing fields, record what was missing)
2. Feature engineering: combined TF-IDF text + numeric meta features + one-hot categoricals
3. Stratified 80/20 train/test split
4. Grid search with 5-fold cross-validation over `C` / `class_weight` (Logistic Regression)
   and `n_estimators` / `max_depth` (Random Forest)
5. Retrain each winner and evaluate on the untouched test set
6. Compare precision, recall and F1 for both models

All the logic lives in `job_fraud_model.py` next to this notebook; the notebook calls it
step by step so each stage's output is visible.

In [1]:
import pandas as pd
import job_fraud_model as jfm

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
print("modules loaded")

modules loaded


---
## 1. Load and clean

Cleaning decisions:

* **`job_id` is dropped** — it is a pure row identifier. Keeping it would add noise at
  best and leak ordering information at worst.
* **Missingness is recorded before it is filled.** Empty fields are not neutral here:
  a scam posting very often has no company profile and no requirements, so
  `company_profile_missing`, `requirements_missing` etc. are real predictive features.
  Once the flags are captured, text is filled with `""` and categories with `"Unknown"`.
* **`location` is collapsed to a country code** (`"US, NY, New York"` → `"US"`). The raw
  field has thousands of distinct values, far too many to one-hot encode usefully.

In [2]:
df = jfm.load_and_clean()


1. LOAD & CLEAN
Raw shape: (17880, 18)

Missing values per column:
salary_range           15012
department             11547
required_education      8105
benefits                7212
required_experience     7050
function                6455
industry                4903
employment_type         3471
company_profile         3308
requirements            2696
location                 346
description                1
title                      0
job_id                     0
telecommuting              0
has_questions              0
has_company_logo           0
fraudulent                 0



Cleaned shape: (17880, 21)  (job_id dropped)
Class balance: 17014 real / 866 fake (4.84% fraudulent)  ->  imbalanced

First 5 cleaned rows:
   text_len  word_count  n_missing_fields country  fraudulent
0      2661         382                 2      US           0
1      6126         901                 1      NZ           0
2      2638         359                 3      US           0
3      5458         709                 1      US           0
4      3863         470                 2      US           0


In [3]:
# First 5 rows of the cleaned dataset
df.head()

,company_profile_missing,description_missing,requirements_missing,benefits_missing,department_missing,salary_range_missing,n_missing_fields,combined_text,text_len,word_count,...,country,employment_type,required_experience,required_education,industry,function,telecommuting,has_company_logo,has_questions,fraudulent
0,0,0,0,1,0,1,2,"Marketing Intern We're Food52, and we've created a groun...",2661,382,...,US,Other,Internship,Unknown,Unknown,Marketing,0,1,0,0
1,0,0,0,0,0,1,1,"Customer Service - Cloud Video Production 90 Seconds, th...",6126,901,...,NZ,Full-time,Not Applicable,Unknown,Marketing and Advertising,Customer Service,0,1,0,0
2,0,0,0,1,1,1,3,Commissioning Machinery Assistant (CMA) Valor Services p...,2638,359,...,US,Unknown,Unknown,Unknown,Unknown,Unknown,0,1,0,0
3,0,0,0,0,0,1,1,Account Executive - Washington DC Our passion for improv...,5458,709,...,US,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0,1,0,0
4,0,0,0,0,1,1,2,Bill Review Manager SpotSource Solutions LLC is a Global...,3863,470,...,US,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0,1,1,0


In [4]:
# The imbalance, stated plainly
counts = df[jfm.TARGET].value_counts().rename({0: "real", 1: "fake"})
print(counts.to_string())
print(f"\nfraudulent rate: {df[jfm.TARGET].mean():.2%}")

fraudulent
real    17014
fake      866

fraudulent rate: 4.84%


---
## 2. Feature engineering

Three blocks are combined in a single `ColumnTransformer`:

| block | source | transform |
|---|---|---|
| **text** | `title` + `company_profile` + `description` + `requirements` + `benefits` merged into `combined_text` | `TfidfVectorizer` (uni+bigrams, sublinear tf, `min_df=3`, 50k features, English stop words) |
| **meta** | `text_len`, `word_count`, `description_len`, `n_missing_fields`, the six `*_missing` flags, `telecommuting`, `has_company_logo`, `has_questions` | `StandardScaler` |
| **categorical** | `employment_type`, `required_experience`, `required_education`, `industry`, `function`, `country` | `OneHotEncoder(handle_unknown="ignore", min_frequency=10)` |

The vectorizer, scaler and encoder all live **inside** the `Pipeline`. This matters: it
means they are fit only on the training folds during cross-validation, so no vocabulary
or scaling statistic ever leaks from validation data into training. `handle_unknown="ignore"`
lets the test set contain categories the training folds never saw.

In [5]:
print("text fields combined :", jfm.TEXT_FIELDS)
print("meta features        :", jfm.META_COLS)
print("one-hot columns      :", jfm.CAT_COLS)

text fields combined : ['title', 'company_profile', 'description', 'requirements', 'benefits']
meta features        : ['text_len', 'word_count', 'description_len', 'n_missing_fields', 'company_profile_missing', 'description_missing', 'requirements_missing', 'benefits_missing', 'department_missing', 'salary_range_missing', 'telecommuting', 'has_company_logo', 'has_questions']
one-hot columns      : ['employment_type', 'required_experience', 'required_education', 'industry', 'function', 'country']


---
## 3. Stratified 80/20 split

`stratify=y` is essential at a 4.8% positive rate — without it, an unlucky random split
can leave the two halves with noticeably different fraud rates and make the test score
meaningless. With it, both halves carry the same proportion of fakes.

The Kaggle file ships as one unsplit CSV, so the split is done here.

In [6]:
X_train, X_test, y_train, y_test = jfm.split_data(df)


3. STRATIFIED 80/20 SPLIT
train: 14,304 rows, 4.84% fraudulent
test : 3,576 rows, 4.84% fraudulent
(stratify=y keeps the positive rate identical in both halves)

First 5 TRAIN rows:
       text_len  n_missing_fields employment_type country
7530       3906                 2       Full-time      US
129        4060                 3       Full-time      US
4640       3272                 3       Full-time      NZ
402        3178                 2       Full-time      US
13218      4119                 1       Full-time      US

First 5 TEST rows:
       text_len  n_missing_fields employment_type country
16995      1685                 5       Full-time      US
9357       2831                 2       Full-time      US
11561      3039                 2        Contract      AU
1105       5688                 1       Full-time      ZA
1980       3303                 2       Full-time      US


---
## 4. Training — grid search with 5-fold cross-validation

For each model every combination in the grid is evaluated 5 times (one per fold) on the
**training set only**; the mean F1 over the 5 folds decides the winner, and
`GridSearchCV` then refits that winner on the whole training set.

### 4a. Logistic Regression — `C` and `class_weight`

* **`C`** is inverse regularisation strength. With ~50k TF-IDF features and 14k rows,
  a small `C` (strong regularisation) guards against overfitting the vocabulary;
  a large `C` lets the model fit rarer terms.
* **`class_weight`** is the lever that matters most on imbalanced data. `None` treats
  every row equally, so the model can minimise loss by mostly predicting "real" —
  high precision, poor recall. `"balanced"` reweights each class inversely to its
  frequency (a fake counts ~20× a real one), which pushes recall up sharply at some
  cost in precision.

In [7]:
search_lr, cv_lr = jfm.train(
    jfm.make_logreg_pipeline(),
    jfm.get_grids(fast=False)["LogisticRegression"],
    X_train, y_train, "LogisticRegression", n_jobs=-1,
)


4. GRID SEARCH + 5-FOLD CV -- LogisticRegression
Grid: {'clf__C': [0.01, 0.1, 1, 10, 100], 'clf__class_weight': [None, 'balanced']}
10 combinations x 5 folds = 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits



All permutations, mean CV F1 over 5 folds (271.1s):
 param_clf__C param_clf__class_weight  mean_test_score  std_test_score  rank_test_score
       100.00                balanced         0.857094        0.007749                1
       100.00                     NaN         0.845885        0.023635                2
        10.00                balanced         0.839676        0.015326                3
        10.00                     NaN         0.819443        0.023199                4
         1.00                balanced         0.717111        0.020053                5
         1.00                     NaN         0.708908        0.014479                6
         0.10                balanced         0.485878        0.012467                7
         0.10                     NaN         0.383959        0.047333                8
         0.01                balanced         0.363354        0.019909                9
         0.01                     NaN         0.084557        0.027

In [8]:
# All permutations with their mean CV F1, best first
cv_lr

,param_clf__C,param_clf__class_weight,mean_test_score,std_test_score,rank_test_score
0,100.00,balanced,0.857094,0.007749,1
1,100.00,NaN,0.845885,0.023635,2
2,10.00,balanced,0.839676,0.015326,3
3,10.00,NaN,0.819443,0.023199,4
4,1.00,balanced,0.717111,0.020053,5
5,1.00,NaN,0.708908,0.014479,6
6,0.10,balanced,0.485878,0.012467,7
7,0.10,NaN,0.383959,0.047333,8
8,0.01,balanced,0.363354,0.019909,9
9,0.01,NaN,0.084557,0.027549,10


In [9]:
print("Best permutation :", search_lr.best_params_)
print(f"Best mean CV F1  : {search_lr.best_score_:.4f}")

Best permutation : {'clf__C': 100, 'clf__class_weight': 'balanced'}
Best mean CV F1  : 0.8571


### Feature engineering traced through 2–3 real examples

Below, three training rows and three test rows are pushed through the fitted
transformer so the raw text, the meta features and the resulting vector are all visible
side by side.

In [10]:
jfm.show_feature_engineering(search_lr.best_estimator_, X_train, "TRAIN", n=3)


FEATURE ENGINEERING TRACE -- 3 examples from TRAIN

--- example 0 ---
  raw combined_text[:160]: 'Contact Center Representatives Tidewater Finance Co. was established in 1992 for the initial purpose of purchasing, and servicing retail installment contracts. '
  meta: text_len=3906, word_count=526, n_missing_fields=2
  cat : employment_type='Full-time', country='US'


  -> vector length 50211 (50000 tf-idf + 13 meta + 198 one-hot)
  -> top tf-idf terms: tidewater=0.120, tidewater finance=0.093, experience collections=0.092, vacation hours=0.090, equivalent customer=0.088, saturdays scheduled=0.088, months employment=0.088, environment equivalent=0.087

--- example 1 ---
  raw combined_text[:160]: 'Customer Service Associate Novitex Enterprise Solutions, formerly Pitney Bowes Management Services, delivers innovative document and communications management s'
  meta: text_len=4060, word_count=527, n_missing_fields=3
  cat : employment_type='Full-time', country='US'
  -> vector length 50211 (50000 tf-idf + 13 meta + 198 one-hot)
  -> top tf-idf terms: audio visual=0.183, conference rooms=0.142, audio=0.137, conference=0.121, rooms=0.116, designated=0.115, equipment=0.100, visual=0.099

--- example 2 ---
  raw combined_text[:160]: 'Automated Test Analyst SilverStripe CMS &amp; Framework is an open source platform of web development tools. The platform is

In [11]:
jfm.show_feature_engineering(search_lr.best_estimator_, X_test, "TEST", n=3)


FEATURE ENGINEERING TRACE -- 3 examples from TEST

--- example 0 ---
  raw combined_text[:160]: 'EXCELLENT ER RN Opportunity Available Now  Our client, located in Urban, IL, is looking for an ER RN to become a member of their team ASAP!Our client is an inst'
  meta: text_len=1685, word_count=241, n_missing_fields=5
  cat : employment_type='Full-time', country='US'
  -> vector length 50211 (50000 tf-idf + 13 meta + 198 one-hot)
  -> top tf-idf terms: rn=0.237, ed=0.195, er=0.177, miles=0.175, enhanced=0.174, relocation=0.124, time hours=0.119, electronic medical=0.119

--- example 1 ---
  raw combined_text[:160]: 'Scrum Master / Website Development Project Manager 352 Inc. is a full-service digital agency creating websites, software and marketing campaigns. We partner wit'
  meta: text_len=2831, word_count=415, n_missing_fields=2
  cat : employment_type='Full-time', country='US'
  -> vector length 50211 (50000 tf-idf + 13 meta + 198 one-hot)
  -> top tf-idf terms: scrum master=0.212, s

### 4b. Random Forest — `n_estimators` and `max_depth`

* **`n_estimators`** is the number of trees. More trees reduce variance and never
  overfit on their own; they just cost time.
* **`max_depth`** caps how deep each tree grows. `None` lets trees grow until pure —
  which on sparse text features is usually what you want, since the signal is spread
  thin across many rare terms. Shallow trees underfit badly here.

The forest gets a lighter vocabulary (20k unigram features) than the linear model: a
tree ensemble cannot exploit 50k sparse columns the way logistic regression can, and
searching them costs a great deal of time for no gain.

In [12]:
search_rf, cv_rf = jfm.train(
    jfm.make_rf_pipeline(),
    jfm.get_grids(fast=False)["RandomForest"],
    X_train, y_train, "RandomForest", n_jobs=1,
)


4. GRID SEARCH + 5-FOLD CV -- RandomForest
Grid: {'clf__n_estimators': [100, 300, 500], 'clf__max_depth': [None, 20, 50]}
9 combinations x 5 folds = 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits



All permutations, mean CV F1 over 5 folds (485.6s):
param_clf__max_depth  param_clf__n_estimators  mean_test_score  std_test_score  rank_test_score
                None                      500         0.781508        0.031125                1
                None                      300         0.776957        0.028276                2
                None                      100         0.772596        0.030978                3
                  50                      100         0.728019        0.030405                4
                  50                      300         0.727969        0.030553                5
                  50                      500         0.725674        0.029968                6
                  20                      100         0.559029        0.049084                7
                  20                      500         0.557820        0.044972                8
                  20                      300         0.550430        0.042298     

In [13]:
cv_rf

,param_clf__max_depth,param_clf__n_estimators,mean_test_score,std_test_score,rank_test_score
0,None,500,0.781508,0.031125,1
1,None,300,0.776957,0.028276,2
2,None,100,0.772596,0.030978,3
3,50,100,0.728019,0.030405,4
4,50,300,0.727969,0.030553,5
5,50,500,0.725674,0.029968,6
6,20,100,0.559029,0.049084,7
7,20,500,0.557820,0.044972,8
8,20,300,0.550430,0.042298,9


In [14]:
print("Best permutation :", search_rf.best_params_)
print(f"Best mean CV F1  : {search_rf.best_score_:.4f}")

Best permutation : {'clf__max_depth': None, 'clf__n_estimators': 500}
Best mean CV F1  : 0.7815


---
## 5. Prediction and evaluation on the held-out test set

The test set has not been touched by any fit so far. Each model is the winning
combination, already refit on the full training set by `GridSearchCV`.

In [15]:
m_lr = jfm.evaluate(search_lr.best_estimator_, X_test, y_test, "LogisticRegression")


5. TEST-SET EVALUATION -- LogisticRegression
Metrics for the FRAUDULENT (positive) class:
  precision = 0.9133
  recall    = 0.9133
  F1        = 0.9133
  (support: ROC-AUC = 0.9940, PR-AUC = 0.9668)

Full classification report:
              precision    recall  f1-score   support

    real (0)     0.9956    0.9956    0.9956      3403
    fake (1)     0.9133    0.9133    0.9133       173

    accuracy                         0.9916      3576
   macro avg     0.9544    0.9544    0.9544      3576
weighted avg     0.9916    0.9916    0.9916      3576

Confusion matrix (rows = actual, cols = predicted):
             pred real  pred fake
actual real       3388         15
actual fake         15        158


In [16]:
m_rf = jfm.evaluate(search_rf.best_estimator_, X_test, y_test, "RandomForest")


5. TEST-SET EVALUATION -- RandomForest
Metrics for the FRAUDULENT (positive) class:
  precision = 1.0000
  recall    = 0.6647
  F1        = 0.7986
  (support: ROC-AUC = 0.9952, PR-AUC = 0.9527)

Full classification report:
              precision    recall  f1-score   support

    real (0)     0.9832    1.0000    0.9916      3403
    fake (1)     1.0000    0.6647    0.7986       173

    accuracy                         0.9838      3576
   macro avg     0.9916    0.8324    0.8951      3576
weighted avg     0.9841    0.9838    0.9822      3576

Confusion matrix (rows = actual, cols = predicted):
             pred real  pred fake
actual real       3403          0
actual fake         58        115


### Sample predictions

The first 5 test-set rows for each model, followed by examples of what each model gets
right and wrong — true positives, false positives and false negatives.

In [17]:
tbl_lr = jfm.sample_predictions(search_lr.best_estimator_, X_test, y_test, "LogisticRegression")


SAMPLE PREDICTIONS -- LogisticRegression
First 5 rows of the test set:
                                                  text_snippet  actual  predicted  P(fake)  correct
16995  EXCELLENT ER RN Opportunity Available Now  Our client,        0          0   0.0137     True
9357   Scrum Master / Website Development Project Manager 352        0          0   0.0005     True
11561  HR Assistant - Contract Squiz is one of the world's lea       0          0   0.0006     True
1105   Regional Sales Director South Africa Upstream’s mission       0          0   0.0000     True
1980   Petrophysicist Valor Services provides Workforce Soluti       0          0   0.0000     True

True positives (caught fakes):
                                                  text_snippet  actual  predicted  P(fake)  correct
7830   Position Finance Assistant  We have positions available       1          1   0.9905     True
17670  Customer Service Agent  We are seeking individuals to b       1          1   0.9997     T

In [18]:
tbl_rf = jfm.sample_predictions(search_rf.best_estimator_, X_test, y_test, "RandomForest")


SAMPLE PREDICTIONS -- RandomForest
First 5 rows of the test set:
                                                  text_snippet  actual  predicted  P(fake)  correct
16995  EXCELLENT ER RN Opportunity Available Now  Our client,        0          0    0.100     True
9357   Scrum Master / Website Development Project Manager 352        0          0    0.004     True
11561  HR Assistant - Contract Squiz is one of the world's lea       0          0    0.002     True
1105   Regional Sales Director South Africa Upstream’s mission       0          0    0.004     True
1980   Petrophysicist Valor Services provides Workforce Soluti       0          0    0.022     True

True positives (caught fakes):
                                                  text_snippet  actual  predicted  P(fake)  correct
17670  Customer Service Agent  We are seeking individuals to b       1          1    0.998     True
5513   Payroll Data Coordinator Positions - Earn $100-$200 Dai       1          1    1.000     True
60

---
## 6. Model comparison

In [19]:
comparison = pd.DataFrame([{
    "model": m["model"],
    "best_params": m["best_params"],
    "cv_f1_mean": round(m["best_cv_f1"], 4),
    "test_precision": round(m["precision_fraudulent"], 4),
    "test_recall": round(m["recall_fraudulent"], 4),
    "test_f1": round(m["f1_fraudulent"], 4),
    "test_roc_auc": round(m["roc_auc"], 4),
    "test_pr_auc": round(m["pr_auc"], 4),
} for m in [
    dict(m_lr, best_params={k: str(v) for k, v in search_lr.best_params_.items()},
         best_cv_f1=search_lr.best_score_),
    dict(m_rf, best_params={k: str(v) for k, v in search_rf.best_params_.items()},
         best_cv_f1=search_rf.best_score_),
]])
comparison

,model,best_params,cv_f1_mean,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,"{'clf__C': '100', 'clf__class_weight': 'balanced'}",0.8571,0.9133,0.9133,0.9133,0.9940,0.9668
1,RandomForest,"{'clf__max_depth': 'None', 'clf__n_estimators': '500'}",0.7815,1.0000,0.6647,0.7986,0.9952,0.9527


In [20]:
winner = comparison.loc[comparison.test_f1.idxmax(), "model"]
print(f"Winner on test F1 (fraudulent class): {winner}")

Winner on test F1 (fraudulent class): LogisticRegression


In [21]:
import matplotlib.pyplot as plt
jfm.plot_confusion_matrices([m_lr, m_rf], jfm.RESULTS_DIR / "confusion_matrices_nb.png")
plt.imshow(plt.imread(jfm.RESULTS_DIR / "confusion_matrices_nb.png"))
plt.axis("off")
plt.gcf().set_size_inches(14, 5)
plt.show()

C:\Users\matan\AppData\Local\Temp\ipykernel_6204\965055131.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Reading the results

The two models fail in opposite directions, which is the interesting part:

* **Logistic Regression with `class_weight="balanced"`** buys recall with precision. It
  catches the large majority of scams but also flags a number of genuine postings.
* **Random Forest** is the conservative one: it flags very few real postings as fake
  (high precision) but lets more scams through (lower recall).

Which is preferable depends on the cost of each mistake. For a job board screening
listings, a missed scam (false negative) harms a job seeker, while a false positive only
costs a human review — that argues for the higher-recall model even at lower F1. F1
itself, the metric the assignment specifies, treats the two errors as equally costly.